In [29]:
# ============================================================
# PERSONALIZED HEALTHCARE SYSTEM
# NOTEBOOK 03 - FINAL UNIFIED MODEL TRAINING
# Dataset 1 + Dataset 3
# Render-Compatible Random Forest
# ============================================================

import os
import re
import gc
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

RANDOM_STATE = 42

# ------------------------------------------------------------
# Find project root robustly
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if (
    (CURRENT_DIR / "data").exists()
    and (CURRENT_DIR / "models").exists()
):
    BASE_DIR = CURRENT_DIR
elif (
    (CURRENT_DIR.parent / "data").exists()
    and (CURRENT_DIR.parent / "models").exists()
):
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR


DATASET_1_PATH = (
    BASE_DIR
    / "data"
    / "dataset_1"
    / "dataset.csv"
)

DATASET_3_PATH = (
    BASE_DIR
    / "data"
    / "dataset_3"
    / "Diseases_and_Symptoms_dataset.csv"
)

MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = MODEL_DIR / "disease_model.pkl"
FEATURE_PATH = MODEL_DIR / "feature_names.pkl"
ENCODER_PATH = MODEL_DIR / "label_encoder.pkl"


print("=" * 80)
print("PERSONALIZED HEALTHCARE SYSTEM")
print("FINAL UNIFIED DISEASE PREDICTION MODEL")
print("=" * 80)

print("Project:", BASE_DIR)
print("Dataset 1:", DATASET_1_PATH)
print("Dataset 3:", DATASET_3_PATH)
print("Model directory:", MODEL_DIR)

PERSONALIZED HEALTHCARE SYSTEM
FINAL UNIFIED DISEASE PREDICTION MODEL
Project: c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System
Dataset 1: c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\data\dataset_1\dataset.csv
Dataset 3: c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\data\dataset_3\Diseases_and_Symptoms_dataset.csv
Model directory: c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\models


In [30]:
# ============================================================
# LOAD DATASETS
# ============================================================

if not DATASET_1_PATH.exists():
    raise FileNotFoundError(
        f"Dataset 1 not found:\n{DATASET_1_PATH}"
    )

if not DATASET_3_PATH.exists():
    raise FileNotFoundError(
        f"Dataset 3 not found:\n{DATASET_3_PATH}"
    )


df1 = pd.read_csv(
    DATASET_1_PATH
)

df3 = pd.read_csv(
    DATASET_3_PATH
)


print("=" * 80)
print("DATASETS LOADED")
print("=" * 80)

print("Dataset 1 shape:", df1.shape)
print("Dataset 3 shape:", df3.shape)

print(
    "Dataset 1 diseases:",
    df1["Disease"].nunique()
)

print(
    "Dataset 3 diseases:",
    df3["diseases"].nunique()
)

DATASETS LOADED
Dataset 1 shape: (4920, 18)
Dataset 3 shape: (96088, 231)
Dataset 1 diseases: 41
Dataset 3 diseases: 100


In [31]:
# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    value = value.replace("_", " ")
    value = value.replace("-", " ")

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def compact_text(value):

    value = normalize_text(value)

    return re.sub(
        r"[^a-z0-9]+",
        "",
        value
    )

In [32]:
# ============================================================
# DISEASE NORMALIZATION
# ============================================================

def normalize_disease(value):

    value = normalize_text(value)

    replacements = {

        "(vertigo) paroymsal positional vertigo":
            "(vertigo) paroxysmal positional vertigo",

        "peptic ulcer diseae":
            "peptic ulcer disease",

        "osteoarthristis":
            "osteoarthritis",
    }

    return replacements.get(
        value,
        value
    )


df1["_disease_normalized"] = (
    df1["Disease"]
    .apply(normalize_disease)
)

df3["_disease_normalized"] = (
    df3["diseases"]
    .apply(normalize_disease)
)


diseases_1 = set(
    df1["_disease_normalized"]
)

diseases_3 = set(
    df3["_disease_normalized"]
)

common_diseases = (
    diseases_1 & diseases_3
)

all_diseases = (
    diseases_1 | diseases_3
)


print("=" * 80)
print("DISEASE ANALYSIS")
print("=" * 80)

print("Dataset 1 diseases:", len(diseases_1))
print("Dataset 3 diseases:", len(diseases_3))
print("Common diseases:", len(common_diseases))
print("Unified diseases:", len(all_diseases))

print("\nCommon diseases:")

for disease in sorted(common_diseases):
    print("-", disease)

DISEASE ANALYSIS
Dataset 1 diseases: 41
Dataset 3 diseases: 100
Common diseases: 8
Unified diseases: 133

Common diseases:
- allergy
- common cold
- drug reaction
- heart attack
- hypoglycemia
- pneumonia
- psoriasis
- urinary tract infection


In [33]:
# ============================================================
# DATASET 3 SYMPTOM VOCABULARY
# ============================================================

dataset3_symptom_columns = list(
    df3.columns[1:]
)

dataset3_normalized_features = {
    normalize_text(column): column
    for column in dataset3_symptom_columns
}


print("=" * 80)
print("DATASET 3 SYMPTOM VOCABULARY")
print("=" * 80)

print(
    "Dataset 3 symptom features:",
    len(dataset3_symptom_columns)
)

DATASET 3 SYMPTOM VOCABULARY
Dataset 3 symptom features: 231


In [34]:
# ============================================================
# DATASET 1 SYMPTOMS
# ============================================================

dataset1_symptom_columns = [
    column
    for column in df1.columns
    if str(column).lower().startswith("symptom")
]


dataset1_symptoms = set()


for column in dataset1_symptom_columns:

    for value in df1[column]:

        normalized = normalize_text(
            value
        )

        if normalized:
            dataset1_symptoms.add(
                normalized
            )


print("=" * 80)
print("DATASET 1 SYMPTOMS")
print("=" * 80)

print(
    "Dataset 1 symptom columns:",
    len(dataset1_symptom_columns)
)

print(
    "Unique Dataset 1 symptoms:",
    len(dataset1_symptoms)
)

DATASET 1 SYMPTOMS
Dataset 1 symptom columns: 17
Unique Dataset 1 symptoms: 131


In [35]:
# ============================================================
# SYMPTOM ALIASES
# ============================================================

SYMPTOM_ALIASES = {

    # Heart
    "fast heart rate":
        "increased heart rate",

    "rapid heart rate":
        "increased heart rate",

    "high heart rate":
        "increased heart rate",

    "heart rate increased":
        "increased heart rate",

    "palpitation":
        "palpitations",

    # Respiratory
    "breathlessness":
        "shortness of breath",

    "difficulty breathing":
        "difficulty breathing",

    "breathing difficulty":
        "difficulty breathing",

    # Skin
    "skin rashes":
        "skin rash",

    "rash":
        "skin rash",

    # Urinary
    "frequent urination":
        "frequent urination",

    "excess urination":
        "excessive urination",

    # Vomiting
    "vomit":
        "vomiting",

    # Head
    "head pain":
        "headache",

    # Joint
    "joint aches":
        "joint pain",
}


def normalize_symptom_name(value):

    value = normalize_text(value)

    return SYMPTOM_ALIASES.get(
        value,
        value
    )

In [36]:
# ============================================================
# DATASET 1 -> DATASET 3 SYMPTOM MATCHING
# ============================================================

matched_symptoms = {}
unmatched_symptoms = []


for symptom in sorted(dataset1_symptoms):

    normalized = normalize_symptom_name(
        symptom
    )

    if normalized in dataset3_normalized_features:

        matched_symptoms[symptom] = (
            dataset3_normalized_features[
                normalized
            ]
        )

    else:

        unmatched_symptoms.append(
            symptom
        )


print("=" * 80)
print("SYMPTOM MATCHING")
print("=" * 80)

print(
    "Dataset 1 unique symptoms:",
    len(dataset1_symptoms)
)

print(
    "Matched to Dataset 3:",
    len(matched_symptoms)
)

print(
    "Still unmatched:",
    len(unmatched_symptoms)
)

print("\nMatched examples:")

for source, target in list(
    matched_symptoms.items()
)[:30]:

    print(
        f"{source} --> {target}"
    )

print("\nUnmatched examples:")

for symptom in unmatched_symptoms[:30]:

    print(symptom)

SYMPTOM MATCHING
Dataset 1 unique symptoms: 131
Matched to Dataset 3: 20
Still unmatched: 111

Matched examples:
back pain --> back pain
breathlessness --> shortness of breath
chills --> chills
constipation --> constipation
cough --> cough
depression --> depression
dizziness --> dizziness
fast heart rate --> increased heart rate
fatigue --> fatigue
headache --> headache
joint pain --> joint pain
knee pain --> knee pain
nausea --> nausea
neck pain --> neck pain
palpitations --> palpitations
restlessness --> restlessness
skin rash --> skin rash
sweating --> sweating
vomiting --> vomiting
weight gain --> weight gain

Unmatched examples:
abdominal pain
abnormal menstruation
acidity
acute liver failure
altered sensorium
anxiety
belly pain
blackheads
bladder discomfort
blister
blood in sputum
bloody stool
blurred and distorted vision
brittle nails
bruising
burning micturition
chest pain
cold hands and feets
coma
congestion
continuous feel of urine
continuous sneezing
cramps
dark urine
dehydr

In [37]:
# ============================================================
# UNIFIED FEATURE SPACE
# ============================================================

unified_features = set(
    dataset3_symptom_columns
)


for symptom in unmatched_symptoms:

    unified_features.add(
        normalize_symptom_name(symptom)
    )


unified_features = sorted(
    unified_features
)


print("=" * 80)
print("UNIFIED FEATURE SPACE")
print("=" * 80)

print(
    "Dataset 3 original features:",
    len(dataset3_symptom_columns)
)

print(
    "Dataset 1 unmatched features added:",
    len(unmatched_symptoms)
)

print(
    "Final unified features:",
    len(unified_features)
)

UNIFIED FEATURE SPACE
Dataset 3 original features: 231
Dataset 1 unmatched features added: 111
Final unified features: 342


In [38]:
# ============================================================
# CONVERT DATASET 3
# ============================================================

X3 = pd.DataFrame(
    0,
    index=np.arange(len(df3)),
    columns=unified_features,
    dtype=np.int8
)


for original_column in dataset3_symptom_columns:

    normalized_column = normalize_symptom_name(
        original_column
    )

    if normalized_column in X3.columns:

        values = pd.to_numeric(
            df3[original_column],
            errors="coerce"
        ).fillna(0)

        X3[normalized_column] = (
            values.astype(np.int8)
            .clip(0, 1)
        )


y3 = df3[
    "_disease_normalized"
].reset_index(drop=True)


X3 = X3.reset_index(drop=True)


print("=" * 80)
print("DATASET 3 CONVERSION")
print("=" * 80)

print("X3:", X3.shape)
print("y3:", y3.shape)

DATASET 3 CONVERSION
X3: (96088, 342)
y3: (96088,)


In [39]:
# ============================================================
# CONVERT DATASET 1
# ============================================================

X1 = pd.DataFrame(
    0,
    index=np.arange(len(df1)),
    columns=unified_features,
    dtype=np.int8
)


for row_index, row in df1.iterrows():

    for column in dataset1_symptom_columns:

        symptom = normalize_symptom_name(
            row[column]
        )

        if not symptom:
            continue

        if symptom in X1.columns:

            X1.loc[
                row_index,
                symptom
            ] = 1


y1 = df1[
    "_disease_normalized"
].reset_index(drop=True)


X1 = X1.reset_index(drop=True)


print("=" * 80)
print("DATASET 1 CONVERSION")
print("=" * 80)

print("X1:", X1.shape)
print("y1:", y1.shape)

DATASET 1 CONVERSION
X1: (4920, 342)
y1: (4920,)


In [40]:
# ============================================================
# COMBINE DATASET 1 + DATASET 3
# ============================================================

X = pd.concat(
    [
        X1,
        X3
    ],
    axis=0,
    ignore_index=True
)

y = pd.concat(
    [
        y1,
        y3
    ],
    axis=0,
    ignore_index=True
)


print("=" * 80)
print("UNIFIED DATASET")
print("=" * 80)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Total records:", len(X))
print("Total diseases:", y.nunique())
print("Total features:", X.shape[1])

UNIFIED DATASET
X shape: (101008, 342)
y shape: (101008,)
Total records: 101008
Total diseases: 133
Total features: 342


In [41]:
# ============================================================
# UNIFIED DATASET VALIDATION
# ============================================================

print("=" * 80)
print("UNIFIED DATASET VALIDATION")
print("=" * 80)


missing_X = X.isna().sum().sum()
missing_y = y.isna().sum()


print("Missing values in X:", missing_X)
print("Missing values in y:", missing_y)


if missing_X != 0:
    raise ValueError(
        "X contains missing values."
    )


if missing_y != 0:
    raise ValueError(
        "y contains missing values."
    )


unique_values = set(
    X.to_numpy().flatten()
)


print(
    "Feature values:",
    sorted(unique_values)
)


if not unique_values.issubset({0, 1}):

    raise ValueError(
        "Features must contain only 0 and 1."
    )


class_counts = y.value_counts()


print(
    "Minimum records for any disease:",
    class_counts.min()
)


if (class_counts < 2).any():

    raise ValueError(
        "Some disease classes have fewer than 2 records."
    )


print("\nValidation PASSED.")

UNIFIED DATASET VALIDATION
Missing values in X: 0
Missing values in y: 0
Feature values: [np.int8(0), np.int8(1)]
Minimum records for any disease: 120

Validation PASSED.


In [42]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y
    )
)


print("=" * 80)
print("TRAIN / TEST SPLIT")
print("=" * 80)

print(
    "Training samples:",
    len(X_train)
)

print(
    "Testing samples:",
    len(X_test)
)

print(
    "Training features:",
    X_train.shape[1]
)

print(
    "Testing features:",
    X_test.shape[1]
)

print(
    "Training diseases:",
    y_train.nunique()
)

print(
    "Testing diseases:",
    y_test.nunique()
)

TRAIN / TEST SPLIT
Training samples: 80806
Testing samples: 20202
Training features: 342
Testing features: 342
Training diseases: 133
Testing diseases: 133


In [43]:
# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()


y_train_encoded = (
    label_encoder.fit_transform(
        y_train
    )
)


y_test_encoded = (
    label_encoder.transform(
        y_test
    )
)


print("=" * 80)
print("LABEL ENCODING")
print("=" * 80)

print(
    "Number of classes:",
    len(label_encoder.classes_)
)

for index, disease in enumerate(
    label_encoder.classes_,
    start=1
):

    print(
        f"{index:03d}. {disease}"
    )

LABEL ENCODING
Number of classes: 133
001. (vertigo) paroxysmal positional vertigo
002. acne
003. actinic keratosis
004. acute bronchiolitis
005. acute bronchitis
006. acute bronchospasm
007. acute kidney injury
008. acute pancreatitis
009. acute sinusitis
010. aids
011. alcoholic hepatitis
012. allergy
013. angina
014. anxiety
015. appendicitis
016. arthritis
017. arthritis of the hip
018. asthma
019. benign prostatic hyperplasia (bph)
020. brachial neuritis
021. bronchial asthma
022. bursitis
023. carpal tunnel syndrome
024. cervical spondylosis
025. chicken pox
026. cholecystitis
027. chronic back pain
028. chronic cholestasis
029. chronic constipation
030. chronic obstructive pulmonary disease (copd)
031. common cold
032. complex regional pain syndrome
033. concussion
034. conjunctivitis
035. conjunctivitis due to allergy
036. contact dermatitis
037. cornea infection
038. croup
039. cystitis
040. degenerative disc disease
041. dengue
042. dental caries
043. depression
044. developm

In [44]:
# ============================================================
# FINAL RANDOM FOREST
# MEMORY CONTROLLED + ACCURACY PRESERVING
# ============================================================

print("=" * 80)
print("TRAINING FINAL MEMORY-CONTROLLED RANDOM FOREST")
print("=" * 80)


model = RandomForestClassifier(

    # Keep enough trees for stable predictions
    n_estimators=30,

    # Main memory-control parameter.
    # Prevents extremely large trees.
    max_leaf_nodes=10000,

    # Additional protection against overgrown branches.
    max_depth=25,

    # Prevent very small terminal nodes.
    min_samples_leaf=1,

    # Same feature selection strategy as original model.
    max_features="sqrt",

    # Preserve class balancing.
    class_weight="balanced",

    random_state=RANDOM_STATE,

    # Use all CPU cores during local training.
    n_jobs=-1
)


print("\nModel configuration:")
print("  Algorithm        : RandomForestClassifier")
print("  Trees            :", model.n_estimators)
print("  Max leaf nodes   :", model.max_leaf_nodes)
print("  Max depth        :", model.max_depth)
print("  Min samples leaf :", model.min_samples_leaf)
print("  Max features     :", model.max_features)
print("  Class weight     :", model.class_weight)


print("\nTraining started...")


model.fit(
    X_train,
    y_train_encoded
)


print("\nTraining completed successfully.")


total_nodes = sum(
    estimator.tree_.node_count
    for estimator in model.estimators_
)


max_tree_depth = max(
    estimator.tree_.max_depth
    for estimator in model.estimators_
)


print("\nActual model structure:")
print(
    "  Trees:",
    len(model.estimators_)
)

print(
    "  Total nodes:",
    total_nodes
)

print(
    "  Maximum actual depth:",
    max_tree_depth
)

TRAINING FINAL MEMORY-CONTROLLED RANDOM FOREST

Model configuration:
  Algorithm        : RandomForestClassifier
  Trees            : 30
  Max leaf nodes   : 10000
  Max depth        : 25
  Min samples leaf : 1
  Max features     : sqrt
  Class weight     : balanced

Training started...

Training completed successfully.

Actual model structure:
  Trees: 30
  Total nodes: 6842
  Maximum actual depth: 25


In [45]:
# ============================================================
# MODEL PERFORMANCE
# ============================================================

y_pred = model.predict(
    X_test
)


accuracy = accuracy_score(
    y_test_encoded,
    y_pred
)


precision = precision_score(
    y_test_encoded,
    y_pred,
    average="weighted",
    zero_division=0
)


recall = recall_score(
    y_test_encoded,
    y_pred,
    average="weighted",
    zero_division=0
)


f1 = f1_score(
    y_test_encoded,
    y_pred,
    average="weighted",
    zero_division=0
)


print("=" * 80)
print("MODEL PERFORMANCE")
print("=" * 80)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

MODEL PERFORMANCE
Accuracy : 0.5119
Precision: 0.6588
Recall   : 0.5119
F1 Score : 0.5465


In [46]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("=" * 80)
print("CLASSIFICATION REPORT")
print("=" * 80)


print(
    classification_report(
        y_test_encoded,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

CLASSIFICATION REPORT
                                              precision    recall  f1-score   support

     (vertigo) paroxysmal positional vertigo       1.00      1.00      1.00        24
                                        acne       1.00      1.00      1.00        24
                           actinic keratosis       0.98      0.75      0.85       162
                         acute bronchiolitis       0.00      0.00      0.00       241
                            acute bronchitis       0.66      0.33      0.44       243
                          acute bronchospasm       0.00      0.00      0.00       162
                         acute kidney injury       0.37      0.75      0.50       162
                          acute pancreatitis       1.00      0.51      0.68       241
                             acute sinusitis       0.59      0.75      0.66       166
                                        aids       1.00      1.00      1.00        24
                         alcoho

In [47]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({

    "Symptom":
        unified_features,

    "Importance":
        model.feature_importances_

})


feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 80)
print("TOP 30 IMPORTANT SYMPTOMS")
print("=" * 80)


display(
    feature_importance.head(30)
)

TOP 30 IMPORTANT SYMPTOMS


,Symptom,Importance
0,yellowing of eyes,0.018155
1,itching,0.016882
2,weight loss,0.016460
3,pain behind the eyes,0.015809
4,mucoid sputum,0.015689
5,mild fever,0.015383
6,fatigue,0.015217
7,lack of concentration,0.015138
8,diarrhoea,0.015105
9,dark urine,0.015052


In [48]:
# ============================================================
# PREDICTION FUNCTION
# ============================================================

def prepare_test_input(symptoms):

    input_data = pd.DataFrame(
        0,
        index=[0],
        columns=unified_features,
        dtype=np.int8
    )


    recognized = []
    unknown = []


    for symptom in symptoms:

        normalized = normalize_symptom_name(
            symptom
        )


        if normalized in input_data.columns:

            input_data.loc[
                0,
                normalized
            ] = 1

            recognized.append(
                normalized
            )

        else:

            unknown.append(
                normalize_text(symptom)
            )


    recognized = list(
        dict.fromkeys(
            recognized
        )
    )


    unknown = list(
        dict.fromkeys(
            unknown
        )
    )


    return (
        input_data,
        recognized,
        unknown
    )


def predict_top_k(
    symptoms,
    k=10
):

    (
        input_data,
        recognized,
        unknown
    ) = prepare_test_input(
        symptoms
    )


    if not recognized:

        raise ValueError(
            "None of the supplied symptoms "
            "exist in the unified feature space."
        )


    probabilities = model.predict_proba(
        input_data
    )[0]


    top_indices = np.argsort(
        probabilities
    )[::-1][:k]


    results = []


    for index in top_indices:

        encoded_class = (
            model.classes_[index]
        )


        disease = (
            label_encoder
            .inverse_transform(
                [encoded_class]
            )[0]
        )


        results.append({

            "Disease":
                disease,

            "Probability":
                float(
                    probabilities[index]
                )

        })


    return (
        pd.DataFrame(results),
        recognized,
        unknown
    )

In [49]:
# ============================================================
# CHEST PAIN TEST
# ============================================================

chest_result, recognized, unknown = (
    predict_top_k(
        ["chest pain"],
        k=10
    )
)


print("=" * 80)
print("CHEST PAIN TEST")
print("=" * 80)

print(
    "Input:",
    ["chest pain"]
)

print(
    "Recognized:",
    recognized
)

print(
    "Unknown:",
    unknown
)


chest_display = (
    chest_result.copy()
)


chest_display["Probability"] = (
    chest_display["Probability"] * 100
).round(2)


display(
    chest_display
)


print("\nTop prediction:")

print(
    chest_display.iloc[0]["Disease"]
)

print(
    "Probability:",
    chest_display.iloc[0]["Probability"],
    "%"
)

CHEST PAIN TEST
Input: ['chest pain']
Recognized: ['chest pain']
Unknown: []


,Disease,Probability
0,gerd,12.05
1,heart attack,8.64
2,hypertension,3.74
3,threatened pregnancy,0.82
4,otitis media,0.81
5,vaginal cyst,0.80
6,croup,0.80
7,sinus bradycardia,0.80
8,conjunctivitis due to allergy,0.79
9,acute bronchospasm,0.79



Top prediction:
gerd
Probability: 12.05 %


In [50]:
# ============================================================
# MULTIPLE SYMPTOM TESTS
# ============================================================

test_cases = {

    "Fast Heart Rate": [
        "fast heart rate"
    ],

    "Diabetes Pattern": [
        "polyuria",
        "weight loss",
        "fatigue"
    ],

    "Abdominal Pain": [
        "abdominal pain"
    ],

    "Pneumonia Pattern": [
        "cough",
        "shortness of breath",
        "chills"
    ]
}


for test_name, symptoms in test_cases.items():

    print()
    print("=" * 80)
    print(test_name)
    print("=" * 80)


    try:

        result, recognized, unknown = (
            predict_top_k(
                symptoms,
                k=5
            )
        )


        print(
            "Input:",
            symptoms
        )

        print(
            "Recognized:",
            recognized
        )

        print(
            "Unknown:",
            unknown
        )


        result_display = (
            result.copy()
        )


        result_display["Probability"] = (
            result_display["Probability"] * 100
        ).round(2)


        display(
            result_display
        )


        print(
            "Top prediction:",
            result_display.iloc[0]["Disease"]
        )


    except Exception as error:

        print(
            "TEST ERROR:",
            error
        )


Fast Heart Rate
Input: ['fast heart rate']
Recognized: ['increased heart rate']
Unknown: []


,Disease,Probability
0,otitis media,1.05
1,croup,1.05
2,acute bronchospasm,1.04
3,heart failure,1.04
4,threatened pregnancy,1.04


Top prediction: otitis media

Diabetes Pattern
Input: ['polyuria', 'weight loss', 'fatigue']
Recognized: ['polyuria', 'weight loss', 'fatigue']
Unknown: []


,Disease,Probability
0,diabetes,35.82
1,jaundice,5.68
2,hyperthyroidism,3.34
3,tuberculosis,3.33
4,hypertensive heart disease,1.68


Top prediction: diabetes

Abdominal Pain
Input: ['abdominal pain']
Recognized: ['abdominal pain']
Unknown: []


,Disease,Probability
0,peptic ulcer disease,10.09
1,jaundice,10.07
2,alcoholic hepatitis,3.36
3,threatened pregnancy,0.81
4,otitis media,0.80


Top prediction: peptic ulcer disease

Pneumonia Pattern
Input: ['cough', 'shortness of breath', 'chills']
Recognized: ['cough', 'shortness of breath', 'chills']
Unknown: []


,Disease,Probability
0,malaria,1.50
1,infectious gastroenteritis,1.38
2,noninfectious gastroenteritis,1.33
3,strep throat,1.24
4,sepsis,1.21


Top prediction: malaria


In [51]:
# ============================================================
# MODEL SIZE CHECK
# ============================================================

import tempfile


TEMP_MODEL_PATH = (
    MODEL_DIR
    / "disease_model_temp.pkl"
)


print("=" * 80)
print("CHECKING MODEL SIZE")
print("=" * 80)


joblib.dump(
    model,
    TEMP_MODEL_PATH,
    compress=3
)


model_size_bytes = (
    TEMP_MODEL_PATH.stat().st_size
)


model_size_mb = (
    model_size_bytes
    / (1024 * 1024)
)


print(
    f"Compressed model size: {model_size_mb:.2f} MB"
)


# ------------------------------------------------------------
# HARD 300 MB LIMIT
# ------------------------------------------------------------

MAX_MODEL_SIZE_MB = 300


if model_size_mb > MAX_MODEL_SIZE_MB:

    TEMP_MODEL_PATH.unlink(
        missing_ok=True
    )

    raise RuntimeError(
        f"""
MODEL TOO LARGE

Current size:
{model_size_mb:.2f} MB

Maximum allowed:
{MAX_MODEL_SIZE_MB} MB

The model was NOT accepted.
"""
    )


print()
print(
    "MODEL SIZE CHECK PASSED."
)

print(
    f"Model is below {MAX_MODEL_SIZE_MB} MB."
)

CHECKING MODEL SIZE
Compressed model size: 0.87 MB

MODEL SIZE CHECK PASSED.
Model is below 300 MB.


In [52]:
# ============================================================
# SAVE FINAL MODEL ARTIFACTS
# ============================================================

print("=" * 80)
print("SAVING FINAL MODEL ARTIFACTS")
print("=" * 80)


# Remove previous final model
if MODEL_PATH.exists():

    MODEL_PATH.unlink()


# Save final model
joblib.dump(
    model,
    MODEL_PATH,
    compress=3
)


# Save features
joblib.dump(
    list(unified_features),
    FEATURE_PATH
)


# Save label encoder
joblib.dump(
    label_encoder,
    ENCODER_PATH
)


# Remove temporary model
TEMP_MODEL_PATH.unlink(
    missing_ok=True
)


print("Model saved:")
print(MODEL_PATH)

print()
print("Features saved:")
print(FEATURE_PATH)

print()
print("Encoder saved:")
print(ENCODER_PATH)

SAVING FINAL MODEL ARTIFACTS
Model saved:
c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\models\disease_model.pkl

Features saved:
c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\models\feature_names.pkl

Encoder saved:
c:\Users\HITESH\OneDrive\Desktop\Personalized_Healthcare_System\models\label_encoder.pkl


In [53]:
# ============================================================
# FINAL ARTIFACT SIZES
# ============================================================

def file_size_mb(path):

    return (
        path.stat().st_size
        / (1024 * 1024)
    )


print("=" * 80)
print("FINAL FILE SIZES")
print("=" * 80)


final_model_size = file_size_mb(
    MODEL_PATH
)

feature_size = (
    FEATURE_PATH.stat().st_size
    / 1024
)

encoder_size = (
    ENCODER_PATH.stat().st_size
    / 1024
)


print(
    f"disease_model.pkl : "
    f"{final_model_size:.2f} MB"
)

print(
    f"feature_names.pkl : "
    f"{feature_size:.2f} KB"
)

print(
    f"label_encoder.pkl : "
    f"{encoder_size:.2f} KB"
)


if final_model_size > 300:

    raise RuntimeError(
        "FINAL MODEL EXCEEDS 300 MB."
    )


print()
print("FINAL SIZE CHECK PASSED.")

FINAL FILE SIZES
disease_model.pkl : 0.87 MB
feature_names.pkl : 6.29 KB
label_encoder.pkl : 2.99 KB

FINAL SIZE CHECK PASSED.


In [54]:
# ============================================================
# FINAL ARTIFACT SIZES
# ============================================================

def file_size_mb(path):

    return (
        path.stat().st_size
        / (1024 * 1024)
    )


print("=" * 80)
print("FINAL FILE SIZES")
print("=" * 80)


final_model_size = file_size_mb(
    MODEL_PATH
)

feature_size = (
    FEATURE_PATH.stat().st_size
    / 1024
)

encoder_size = (
    ENCODER_PATH.stat().st_size
    / 1024
)


print(
    f"disease_model.pkl : "
    f"{final_model_size:.2f} MB"
)

print(
    f"feature_names.pkl : "
    f"{feature_size:.2f} KB"
)

print(
    f"label_encoder.pkl : "
    f"{encoder_size:.2f} KB"
)


if final_model_size > 300:

    raise RuntimeError(
        "FINAL MODEL EXCEEDS 300 MB."
    )


print()
print("FINAL SIZE CHECK PASSED.")

FINAL FILE SIZES
disease_model.pkl : 0.87 MB
feature_names.pkl : 6.29 KB
label_encoder.pkl : 2.99 KB

FINAL SIZE CHECK PASSED.


In [55]:
# ============================================================
# RELOADED MODEL PREDICTION TEST
# ============================================================

def predict_with_loaded_model(
    symptoms,
    k=5
):

    input_data = pd.DataFrame(
        0,
        index=[0],
        columns=loaded_features,
        dtype=np.int8
    )


    recognized = []
    unknown = []


    for symptom in symptoms:

        normalized = normalize_symptom_name(
            symptom
        )


        if normalized in input_data.columns:

            input_data.loc[
                0,
                normalized
            ] = 1

            recognized.append(
                normalized
            )

        else:

            unknown.append(
                normalize_text(symptom)
            )


    if not recognized:

        raise ValueError(
            "No recognized symptoms."
        )


    probabilities = (
        loaded_model
        .predict_proba(
            input_data
        )[0]
    )


    indices = np.argsort(
        probabilities
    )[::-1][:k]


    output = []


    for index in indices:

        encoded_class = (
            loaded_model.classes_[index]
        )


        disease = (
            loaded_encoder
            .inverse_transform(
                [encoded_class]
            )[0]
        )


        output.append({

            "Disease":
                disease,

            "Probability":
                float(
                    probabilities[index]
                )

        })


    return pd.DataFrame(output), recognized, unknown


# ------------------------------------------------------------
# Final Chest Pain verification
# ------------------------------------------------------------

final_chest, recognized, unknown = (
    predict_with_loaded_model(
        ["chest pain"],
        k=5
    )
)


final_chest["Probability"] = (
    final_chest["Probability"] * 100
).round(2)


print("=" * 80)
print("FINAL RELOADED MODEL - CHEST PAIN")
print("=" * 80)

print(
    "Recognized:",
    recognized
)

print(
    "Unknown:",
    unknown
)

display(
    final_chest
)


print(
    "\nFINAL CHEST PAIN PREDICTION:",
    final_chest.iloc[0]["Disease"]
)

print(
    "Probability:",
    final_chest.iloc[0]["Probability"],
    "%"
)

FINAL RELOADED MODEL - CHEST PAIN
Recognized: ['chest pain']
Unknown: []


,Disease,Probability
0,gum disease,0.96
1,eczema,0.96
2,dental caries,0.96
3,vaginal cyst,0.96
4,threatened pregnancy,0.96



FINAL CHEST PAIN PREDICTION: gum disease
Probability: 0.96 %


In [57]:
# ============================================================
# RAW CHEST PAIN PROBABILITY DEBUG
# ============================================================

import numpy as np
import pandas as pd

test_input = np.zeros((1, len(unified_features)), dtype=np.int8)

chest_idx = unified_features.index("chest pain")
test_input[0, chest_idx] = 1

raw_probs = model.predict_proba(test_input)[0]

classes = model.classes_
diseases = label_encoder.inverse_transform(classes)

debug_df = pd.DataFrame({
    "Disease": diseases,
    "RawProbability": raw_probs
}).sort_values("RawProbability", ascending=False).reset_index(drop=True)

print("=" * 80)
print("RAW MODEL PROBABILITIES - CHEST PAIN")
print("=" * 80)

print(debug_df.head(15).to_string(index=False))

print("\nSUM OF ALL PROBABILITIES:", raw_probs.sum())
print("MAX RAW PROBABILITY:", raw_probs.max())
print("TOP RAW DISEASE:", diseases[np.argmax(raw_probs)])

RAW MODEL PROBABILITIES - CHEST PAIN
                       Disease  RawProbability
                          gerd        0.120509
                  heart attack        0.086441
                  hypertension        0.037408
          threatened pregnancy        0.008166
                  otitis media        0.008068
                  vaginal cyst        0.008018
                         croup        0.007998
             sinus bradycardia        0.007953
 conjunctivitis due to allergy        0.007948
            acute bronchospasm        0.007947
                 heart failure        0.007941
           acute bronchiolitis        0.007914
                conjunctivitis        0.007911
complex regional pain syndrome        0.007898
                  appendicitis        0.007884

SUM OF ALL PROBABILITIES: 1.0
MAX RAW PROBABILITY: 0.12050930246393218
TOP RAW DISEASE: gerd


In [60]:
# ============================================================
# CHEST PAIN + HEART ATTACK SYMPTOMS TEST
# ============================================================

tests = [
    ["chest pain", "shortness of breath"],
    ["chest pain", "sweating"],
    ["chest pain", "shortness of breath", "sweating"],
    ["chest pain", "shortness of breath", "sweating", "increased heart rate"],
]

for symptoms in tests:
    result = predict_top_k(symptoms)

    print("\n" + "=" * 80)
    print("INPUT:", symptoms)
    print("=" * 80)

    # predict_top_k returns a tuple
    if isinstance(result, tuple):
        prediction_df = result[0]
    else:
        prediction_df = result

    print(prediction_df.head(5).to_string(index=False))


INPUT: ['chest pain', 'shortness of breath']
             Disease  Probability
        heart attack     0.177944
                gerd     0.062339
threatened pregnancy     0.008166
        otitis media     0.008068
        vaginal cyst     0.008018

INPUT: ['chest pain', 'sweating']
     Disease  Probability
heart attack     0.148866
        gerd     0.087339
hypertension     0.037395
hypoglycemia     0.020882
   pneumonia     0.019117

INPUT: ['chest pain', 'shortness of breath', 'sweating']
     Disease  Probability
heart attack     0.240369
        gerd     0.029169
hypoglycemia     0.020882
   pneumonia     0.019117
      angina     0.013415

INPUT: ['chest pain', 'shortness of breath', 'sweating', 'increased heart rate']
             Disease  Probability
        heart attack     0.243033
                gerd     0.029169
              angina     0.028152
           pneumonia     0.019117
threatened pregnancy     0.007446


In [61]:
# ============================================================
# FINAL MODEL SUMMARY
# ============================================================

print("=" * 80)
print("FINAL UNIFIED MODEL SUMMARY")
print("=" * 80)


print(
    "Dataset 1 records:",
    len(df1)
)

print(
    "Dataset 3 records:",
    len(df3)
)

print(
    "Combined records:",
    len(X)
)

print(
    "Common diseases:",
    len(common_diseases)
)

print(
    "Final disease classes:",
    len(label_encoder.classes_)
)

print(
    "Final symptom features:",
    len(unified_features)
)

print(
    "Training records:",
    len(X_train)
)

print(
    "Testing records:",
    len(X_test)
)

print(
    "Accuracy:",
    f"{accuracy:.4f}"
)

print(
    "Precision:",
    f"{precision:.4f}"
)

print(
    "Recall:",
    f"{recall:.4f}"
)

print(
    "F1:",
    f"{f1:.4f}"
)

print(
    "Final model size:",
    f"{final_model_size:.2f} MB"
)


print()
print("FINAL FILES:")

print("1. disease_model.pkl")
print("2. feature_names.pkl")
print("3. label_encoder.pkl")


print()
print("=" * 80)
print("FINAL MODEL TRAINING COMPLETED")
print("=" * 80)

FINAL UNIFIED MODEL SUMMARY
Dataset 1 records: 4920
Dataset 3 records: 96088
Combined records: 101008
Common diseases: 8
Final disease classes: 133
Final symptom features: 342
Training records: 80806
Testing records: 20202
Accuracy: 0.5119
Precision: 0.6588
Recall: 0.5119
F1: 0.5465
Final model size: 0.87 MB

FINAL FILES:
1. disease_model.pkl
2. feature_names.pkl
3. label_encoder.pkl

FINAL MODEL TRAINING COMPLETED


In [62]:
# ============================================================
# FINAL MODEL VERIFICATION
# ============================================================

import os
from pathlib import Path

print("=" * 80)
print("FINAL MODEL VERIFICATION")
print("=" * 80)

# Model size
size_mb = MODEL_PATH.stat().st_size / (1024 * 1024)
print(f"\nFinal model size: {size_mb:.2f} MB")

# Accuracy / F1
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

# Model information
print(f"Features: {len(unified_features)}")
print(f"Disease classes: {len(label_encoder.classes_)}")

# Chest pain final test
chest_result = predict_top_k(["chest pain"])
print("\nFINAL CHEST PAIN:")
print(chest_result[0].head(5).to_string(index=False))

# Combination test
combo_result = predict_top_k(
    ["chest pain", "shortness of breath", "sweating", "increased heart rate"]
)

print("\nFINAL CHEST PAIN COMBINATION:")
print(combo_result[0].head(5).to_string(index=False))

print("\n" + "=" * 80)
print("VERIFICATION COMPLETE")
print("=" * 80)

FINAL MODEL VERIFICATION

Final model size: 0.87 MB
Accuracy: 0.5119
F1 Score: 0.5465
Features: 342
Disease classes: 133

FINAL CHEST PAIN:
             Disease  Probability
                gerd     0.120509
        heart attack     0.086441
        hypertension     0.037408
threatened pregnancy     0.008166
        otitis media     0.008068

FINAL CHEST PAIN COMBINATION:
             Disease  Probability
        heart attack     0.243033
                gerd     0.029169
              angina     0.028152
           pneumonia     0.019117
threatened pregnancy     0.007446

VERIFICATION COMPLETE
